# Nanopore3 in Colab

Demultiplex Oxford Nanopore amplicon reads by plate and well barcode, assign them to
designed references, build one consensus per clone, and grade the result.

**Two paths through this notebook.** Run *Setup*, then either:

- **A — Try it** on the synthetic example that ships with the package. Takes seconds,
  needs no data of your own, and produces the same outputs a real run does.
- **B — Your own data** from Google Drive. Section B stands on its own; you do not have
  to run section A first.

### What Colab is and is not good for here

| | |
|---|---|
| trying the tool, teaching it, onboarding | **ideal** — install is one cell, two dependencies |
| checking barcode recovery on a fresh flowcell | **good** — subsample 100k reads, ~2 minutes |
| a full multi-gigabyte production run | **use a local machine** — see the note in section B |

The constraint is never the alignment; it is moving gigabytes through the Drive mount
on 2 vCPUs. Section B says how to work with that.


## Setup


In [ ]:
# Two pure-Python dependencies (edlib, PyYAML), no compiler, no conda.
%pip install --quiet "git+https://github.com/t-j-fryer/nanopore3.git"

# Optional: figures in the HTML report. Everything else works without it, and a
# missing matplotlib is recorded as skipped rather than failing the run.
%pip install --quiet matplotlib pandas


If the install fails with a 404 or an authentication prompt, the repository is still
private. Either open it, or upload a built wheel to Drive and
`%pip install /content/drive/MyDrive/nanopore3-0.3.0-py3-none-any.whl` instead.


In [ ]:
# Imports used throughout, so either section below can be run on its own.
import os, shutil, time
from pathlib import Path

import pandas as pd


In [ ]:
!nanopore3 doctor

# What this session actually got. Colab's allocation varies, and `jobs: 0` in a
# configuration means 'use whatever is here', so this is what will be used.
print(f'\nCPUs available: {os.cpu_count()}')
!free -h 2>/dev/null | head -2 || true
!df -h /content | tail -1


`doctor` reports the runtime and any optional native backends. `mafft`, `spoa` and
`minimap2` showing as *not found* is expected and fine — the portable `edlib` backend
needs none of them.


---
## A — Try it on the synthetic example


In [ ]:
!nanopore3 init /content/example
!nanopore3 run --config /content/example/configs/example.yaml --output /content/runs --run-id demo

# A failing `!command` prints its error but does not stop the notebook, which
# would leave every cell below reporting a confusing missing file.
index = Path('/content/runs/demo/consensus_by_plate/index.csv')
assert index.is_file(), f'the run did not complete - see the error above'


Each stage reports as it runs. The three outputs named at the end are the ones worth
opening; `index.csv` is where analysis normally starts.


In [ ]:
clones = pd.read_csv('/content/runs/demo/consensus_by_plate/index.csv')
print(clones['grade'].value_counts().to_string())
clones.head()


---
## B — Your own data

### Before you start: the one thing that matters

Colab's bottleneck is the Drive mount, not the pipeline. The input is read about three
times (checksum, record scan, ingest), so an 8 GB FASTQ read straight from Drive spends
most of the run on I/O.

**Copy the input to `/content` first.** One Drive read, then everything is local disk.
Write outputs locally too and copy back only what you need — the graded tree is thousands
of small files, which is the worst case for Drive writes.

If your FASTQ is more than a few GB, run it locally instead. This section is for
moderate runs and for subsampled checks.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


### Point the configuration at this machine

Configurations use `${VARIABLE}` for anything machine-specific, so the same file works
here and on a workstation. Set them for this session:


In [ ]:
# Adjust these two to your Drive layout.
os.environ['SEQ_DATA'] = '/content/drive/MyDrive/sequencing'
os.environ['REFS'] = '/content/drive/MyDrive/references'

for name in ('SEQ_DATA', 'REFS'):
    where = Path(os.environ[name])
    print(f'{name:9s} {str(where):55s} exists={where.is_dir()}')

# An empty or unset variable is refused with the variable named, rather than
# quietly producing a path like '/my_run.fastq'.
assert all(os.environ[n].strip() for n in ('SEQ_DATA', 'REFS')), 'set both paths above'


### Copy the input locally, and optionally subsample it

Set `SUBSAMPLE` to a read count for a quick check, or `None` to use the whole file.


In [ ]:
SOURCE = Path(os.environ['SEQ_DATA']) / 'my_run.fastq'   # <- your FASTQ
SUBSAMPLE = 100_000                                      # or None for all reads

assert SOURCE.is_file(), f'not found: {SOURCE}'
local = Path('/content/input.fastq')
started = time.time()
if SUBSAMPLE:
    # Reads the head of the file only, so this is cheap even on a large input.
    # Both paths are quoted: Drive folder names often contain spaces.
    !nanopore3 subsample --input "{SOURCE}" --output "{local}" --reads {SUBSAMPLE}
else:
    shutil.copy2(SOURCE, local)
    print(f'copied {local.stat().st_size / 1e9:.2f} GB')
print(f'{time.time() - started:.0f}s')


### Write the configuration

This is example 1a from [the worked examples](https://github.com/t-j-fryer/nanopore3/blob/main/docs/workflows.md)
— designed inserts, one culture plate per barcode. That page has the other three:
reconstructing the constant regions, already-assembled references, and pooled colony PCR.

`preset: ont-r10-amplicon` supplies the ~22 tuning values nobody can set from first
principles. `jobs: 0` means *use whatever this machine has*, so the same file runs on
Colab's 2 cores and on a workstation's 16 without editing.


In [ ]:
config = Path('/content/run.yaml')
config.write_text('''schema_version: 1
preset: ont-r10-amplicon
run_name: colab-run
output_root: /content/runs

inputs:
  - path: /content/input.fastq
    sample_id: run1

reference_libraries:
  designs:
    fasta: ${REFS}/my_designs.fasta

plate_reference_map:
  BC01: designs

library:
  name: my-amplicon
  forward_motif: CATAATCCGCACGCATCTGG
  reverse_motif: CGGATTGGCGAATGGGACGC
  minimum_read_length: 300
  maximum_read_length: 1200

barcodes:
  plate:
    registry_csv: ${REFS}/plate_barcodes.csv
    family_id: my_plates
  well:
    registry_csv: ${REFS}/well_barcodes.csv
    family_id: my_wells

parallel:
  jobs: 0

random_seed: 1
''')
print(config.read_text())


### Check it before spending the time

`validate` resolves every path, reads the references, checks the barcode panel is
distinguishable and — if you configured pooling — cross-checks the layout against your
reference names. It is much cheaper than finding out at stage five.


In [ ]:
!nanopore3 validate --config /content/run.yaml --quick


In [ ]:
!nanopore3 run --config /content/run.yaml --run-id colab

# A failing `!command` prints its error but does not stop the notebook, which
# would leave every cell below reporting a confusing missing file.
index = Path('/content/runs/colab/consensus_by_plate/index.csv')
assert index.is_file(), f'the run did not complete - see the error above'


### Read the results


In [ ]:
run = Path('/content/runs/colab')
clones = pd.read_csv(run / 'consensus_by_plate' / 'index.csv')
print(clones['grade'].value_counts().to_string())

# For a mixed clone: how much of the well still carries the designed sequence.
mixed = clones[clones['grade'].str.startswith('mixed')]
columns = ['well_id', 'design', 'grade', 'mixed_worst_effect', 'designed_allele_fraction']
mixed[[c for c in columns if c in mixed.columns]] if len(mixed) else 'no mixed clones'


### Copy back only what is worth keeping

The whole run directory can be gigabytes and is mostly intermediates. The graded tree,
the QC table and the report are tens of megabytes.


In [ ]:
OUT = Path('/content/drive/MyDrive/nanopore3_results')
OUT.mkdir(parents=True, exist_ok=True)

if (run / 'consensus_by_plate').is_dir():
    shutil.make_archive(str(OUT / 'clones'), 'zip', run / 'consensus_by_plate')

# run.json is the provenance record: resolved config, input checksums, versions.
for relative in ('stages/05_qc/qc.csv.gz', 'stages/06_report/report.html', 'run.json'):
    source = run / relative
    if source.is_file():
        shutil.copy2(source, OUT / source.name)
    else:
        print(f'skipped (absent): {relative}')

print(*sorted(p.name for p in OUT.iterdir()), sep='\n')


---
## If the session drops mid-run

Colab disconnects on idleness and has a session cap, and a large run can outlast both.
You do not have to start again — as long as the run directory survived, carry the
finished stages forward:

```
!nanopore3 rerun --config /content/run.yaml \
    --from-run /content/runs/colab --from 03_assignment --run-id colab2
```

Stages before `--from` are inherited by hard link; everything after is recomputed. The
input FASTQ is not even needed once demultiplexing is done. A stage is only inherited if
the current configuration reproduces the fingerprint it recorded, so a changed setting is
refused by name rather than silently built upon.

`/content` is ephemeral, so to survive a disconnect at all, copy the run directory to
Drive periodically:

```python
shutil.make_archive('/content/drive/MyDrive/run_backup', 'zip', run)
```

---
## Notes

- **Figures** fall back to DejaVu Sans; Arial is not installed on Colab, so the house
  style will not match a local render.
- **Reproducibility** is unaffected by running here. `run.json` records the resolved
  configuration, input checksums, package versions and the platform.
- **Documentation**: [worked examples](https://github.com/t-j-fryer/nanopore3/blob/main/docs/workflows.md),
  [references](https://github.com/t-j-fryer/nanopore3/blob/main/docs/references.md),
  [pooling layout](https://github.com/t-j-fryer/nanopore3/blob/main/docs/pooling-layout.md).
